# Module 2: Sequential Decision-Making Agent  
## MDPs, Bandits, Partial Observability, and Approximate POMDPs

This notebook develops a sequential decision-making model in a campus environment and extends it across several increasingly complex settings.

The overall idea is to model an intelligent campus agent that must make decisions over time while facing uncertainty, delayed consequences, and incomplete information. The notebook begins by defining the world and estimating probabilities of crowding from Monte Carlo simulation. It then uses those probabilities in several decision-making frameworks.

The progression of the notebook is:

1. **Campus domain and uncertainty model**  
   The campus is represented as a graph of locations connected by paths. Student movement patterns are simulated over time to estimate how likely each location is to be busy at each time slot.

2. **Sequential decision problem and MDP solution**  
   The agent reasons over states that include time, location, and energy. Actions affect future states, so good decisions must consider long-term consequences rather than immediate reward only.

3. **Bandit learning under unknown rewards**  
   A simpler one-step version of the problem is turned into a multi-armed bandit. This allows comparison of greedy, ε-greedy, and UCB1 strategies to study exploration versus exploitation.

4. **Partial observability and belief states**  
   The true crowd condition of StudentCenter is hidden. The agent cannot directly observe it, so it must maintain and update a belief using Bayes’ rule after receiving noisy observations.

5. **Approximate POMDP planning**  
   Since exact POMDP solving is computationally difficult, the notebook uses rollout-based approximation to estimate action values under uncertainty.

This notebook therefore demonstrates several key ideas from sequential decision-making:
- planning over time,
- probabilistic reasoning,
- expected reward optimization,
- exploration versus exploitation,
- belief-state updates,
- and approximate decision-making in partially observable environments.

In [1]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
import random
import math
from collections import Counter, defaultdict

# ----------------------------
# Domain constants 
# ----------------------------

LOCATIONS = [
    "LawrenceHall",
    "ThayerHall",
    "WestPenn",
    "StudentCenter",
    "Library",
    "VillagePark",
    "PlayHouse",
    "BoulevardStudios",
    "BoulevardAppartments",
    "OffCampus",
]

TIME_SLOTS = ["08:00", "10:00", "12:00", "14:00", "16:00", "18:00", "21:00"]
NUM_TIME_STEPS = len(TIME_SLOTS)

AGENT_TYPE = "athlete"  # for demo its possible to change to "copa" or "regular"


# ----------------------------
# Campus Graph (B option)
# ----------------------------
# Only allow moving along edges. This is what makes the problem truly sequential.

GRAPH: Dict[str, List[str]] = {
    "LawrenceHall": ["StudentCenter", "Library", "ThayerHall"],
    "ThayerHall": ["LawrenceHall", "WestPenn", "BoulevardStudios"],
    "WestPenn": ["ThayerHall", "VillagePark", "OffCampus"],
    "StudentCenter": ["LawrenceHall", "Library", "PlayHouse", "BoulevardAppartments"],
    "Library": ["LawrenceHall", "StudentCenter", "BoulevardStudios"],
    "VillagePark": ["WestPenn", "StudentCenter", "OffCampus"],
    "PlayHouse": ["StudentCenter", "BoulevardStudios"],
    "BoulevardStudios": ["Library", "PlayHouse", "ThayerHall", "BoulevardAppartments"],
    "BoulevardAppartments": ["StudentCenter", "BoulevardStudios", "OffCampus"],
    "OffCampus": ["WestPenn", "VillagePark", "BoulevardAppartments"],
}

def neighbors(loc: str) -> List[str]:
    return GRAPH.get(loc, [])


# ----------------------------
# Energy model (creates "greedy fails")
# ----------------------------
# Energy ∈ {0,1,2} = Low, OK, High
LOW, OK, HIGH = 0, 1, 2

def clamp_energy(e: int) -> int:
    return max(LOW, min(HIGH, e))

## Part 1 — Campus World, Graph Structure, and Hidden Crowd Uncertainty

This section defines the environment in which all later decision-making takes place.

The campus is represented as a set of named locations connected by a graph. This matters because the agent cannot simply jump from any place to any other place in one step. Instead, it can only move along valid graph edges. That makes the problem genuinely sequential: where the agent is now constrains what actions are possible next.

In addition to location and time, the model also includes an **energy variable** with three levels:
- Low
- OK
- High

This variable is important because actions have future consequences. Moving tends to reduce energy, while resting helps recover it. Because energy affects future rewards, an action that looks attractive now may become harmful later. This is the core reason why a greedy strategy can fail.

The section also sets up the overall domain constants used throughout the notebook:
- the list of campus locations,
- the time slots across the day,
- the graph of allowed movements,
- and the energy model.

Together, these define the state space that the agent must reason about.

In [2]:
# ----------------------------
# "Crowd / Busy" belief table (derived from Week1-style Monte Carlo)
# ----------------------------
# We estimate p_busy[L,t] = P( Busy(L,t) ), Busy = frac(students at L) > threshold
# This keeps the Week 1 spirit: uncertainty about crowdiness.

STUDENT_TYPES = ["athlete", "copa", "regular"]
ATHLETE_PROGRAMS = [
    "Baseball", "TrackAndField", "Wrestling", "Soccer", "ESports",
    "Softball", "Lacrosse", "Basketball", "Volleyball", "Golf"
]
COPA_PROGRAMS = [
    "Dance", "MusicalTheater", "Acting", "Film", "Animation", "ScreenWriting"
]

STUDENT_TYPE_PREFERENCES = {
    "athlete": [
        ["VillagePark", "StudentCenter"],                 # 08:00
        ["StudentCenter", "Library"],                     # 10:00
        ["StudentCenter", "BoulevardAppartments"],        # 12:00
        ["Library", "ThayerHall"],                        # 14:00
        ["VillagePark", "WestPenn"],                      # 16:00
        ["StudentCenter", "OffCampus"],                   # 18:00
        LOCATIONS,                                        # 21:00
    ],
    "copa": [
        ["PlayHouse", "BoulevardStudios"],                # 08:00
        ["BoulevardStudios", "PlayHouse"],                # 10:00
        ["StudentCenter", "Library"],                     # 12:00
        ["BoulevardStudios", "ThayerHall"],               # 14:00
        ["PlayHouse", "StudentCenter"],                   # 16:00
        ["StudentCenter", "OffCampus"],                   # 18:00
        LOCATIONS,                                        # 21:00
    ],
    "regular": [LOCATIONS] * NUM_TIME_STEPS,
}

PROGRAM_BOOST = {
    # athlete examples
    "TrackAndField": ["VillagePark"],
    "Soccer": ["VillagePark"],
    "Lacrosse": ["VillagePark"],
    "Baseball": ["VillagePark"],
    "Golf": ["VillagePark"],
    "Basketball": ["StudentCenter"],
    "Volleyball": ["StudentCenter"],
    "ESports": ["BoulevardAppartments", "StudentCenter"],
    # copa examples
    "Dance": ["BoulevardStudios"],
    "MusicalTheater": ["PlayHouse"],
    "Acting": ["PlayHouse"],
    "Film": ["PlayHouse", "BoulevardStudios"],
    "Animation": ["BoulevardStudios"],
    "ScreenWriting": ["Library"],
}

@dataclass(frozen=True)
class Person:
    sid: str
    stype: str
    program: str
    friends: Tuple[str, ...]  # immutable for copying safety

def make_population(rng: random.Random) -> List[Person]:
    # Fixed size population (stable estimates)
    # Feel free to increase counts for better estimates.
    pop: List[Person] = []

    # athletes
    for i in range(6):
        prog = rng.choice(ATHLETE_PROGRAMS)
        pop.append(Person(sid=f"a{i}", stype="athlete", program=prog, friends=tuple()))
    # copa
    for i in range(4):
        prog = rng.choice(COPA_PROGRAMS)
        pop.append(Person(sid=f"c{i}", stype="copa", program=prog, friends=tuple()))
    # regular
    for i in range(10):
        pop.append(Person(sid=f"r{i}", stype="regular", program="None", friends=tuple()))

    # make simple friends (same type) to keep relational influence
    ids_by_type: Dict[str, List[str]] = defaultdict(list)
    for p in pop:
        ids_by_type[p.stype].append(p.sid)

    # rebuild with friends (immutably)
    pop2: List[Person] = []
    for p in pop:
        candidates = [x for x in ids_by_type[p.stype] if x != p.sid]
        k = rng.randint(0, min(2, len(candidates))) if candidates else 0
        friends = tuple(rng.sample(candidates, k)) if k else tuple()
        pop2.append(Person(p.sid, p.stype, p.program, friends))

    return pop2

def choose_location_for_person(
    person: Person,
    time_step: int,
    prev_positions: Dict[str, str],
    friend_boost: int,
    rng: random.Random,
) -> str:
    base = list(STUDENT_TYPE_PREFERENCES[person.stype][time_step])

    boosted: List[str] = []
    # friends
    for fid in person.friends:
        if fid in prev_positions:
            boosted.append(prev_positions[fid])
    # program
    boosted += PROGRAM_BOOST.get(person.program, [])

    possible = base + boosted * friend_boost if boosted else base
    return rng.choice(possible)

def simulate_campus_day_positions(
    population: List[Person],
    friend_boost: int,
    rng: random.Random,
) -> List[Dict[str, str]]:
    trace: List[Dict[str, str]] = []
    prev: Dict[str, str] = {}

    for t in range(NUM_TIME_STEPS):
        step: Dict[str, str] = {}
        order = population[:]
        rng.shuffle(order)
        for p in order:
            step[p.sid] = choose_location_for_person(p, t, prev, friend_boost, rng)
        trace.append(step)
        prev = step

    return trace

def estimate_p_busy_table(
    num_traces: int = 2000,
    busy_threshold: float = 0.18,
    friend_boost: int = 2,
    seed: int = 7,
) -> List[Dict[str, float]]:
    """
    Returns:
      p_busy_table[t][L] = P(Busy(L,t)=True)
    Busy(L,t)=True if frac(all students at L at t) > busy_threshold
    """
    rng = random.Random(seed)
    busy_counts: List[Counter] = [Counter() for _ in range(NUM_TIME_STEPS)]
    totals = [0 for _ in range(NUM_TIME_STEPS)]

    for _ in range(num_traces):
        pop = make_population(rng)
        tr = simulate_campus_day_positions(pop, friend_boost, rng)

        for t in range(NUM_TIME_STEPS):
            step = tr[t]
            n = max(1, len(step))
            counts = Counter(step.values())
            for L in LOCATIONS:
                frac = counts[L] / n
                if frac > busy_threshold:
                    busy_counts[t][L] += 1
            totals[t] += 1

    p_busy_table: List[Dict[str, float]] = []
    for t in range(NUM_TIME_STEPS):
        p_busy_table.append({L: (busy_counts[t][L] / totals[t]) for L in LOCATIONS})

    return p_busy_table

## Part 2 — Building Beliefs About the Campus from Monte Carlo Simulation

Before the agent can make rational decisions, it needs some probabilistic model of the environment. This section builds that model.

The core uncertainty here is whether a location is **busy** at a given time. Rather than hard-coding those probabilities, the notebook estimates them from simulation. A population of students is generated, each with:
- a student type,
- a program,
- and optional friend influence.

Their movement across the day is then simulated many times. From those simulated traces, the notebook estimates:

\[
p\_busy[L,t] = P(Busy(L,t))
\]

where a location is considered busy if the fraction of all students there exceeds a threshold.

This is an important bridge between the earlier probabilistic campus simulation and the later decision-making model:
- the simulation provides uncertainty estimates,
- and the decision-making algorithms use those estimates to compute expected reward.

So this section is not yet choosing actions. Instead, it is learning the probabilistic background of the world that later decisions depend on.

In [3]:
# ============================================================
# Part 1 (17.1): Sequential Decision Problem (why greedy fails)
# ============================================================

"""
We will define a sequential problem:
- State s = (t, loc, energy)
- Actions a:
  - MoveTo(nbr) for neighbor nbr in GRAPH[loc]
  - Rest (stay; improves energy)
  - (optional) CheckCrowd(StudentCenter) for POMDP parts

Greedy can fail because:
- At 16:00 it may take a "fun" action that drains energy,
  but then at 18:00 (practice) you get penalized heavily if energy is Low.
The optimal policy may rest earlier to be ready later.
"""


# ============================================================
# Part 2 (17.2): MDP + Value Iteration
# ============================================================

State = Tuple[int, str, int]  # (time_step, location, energy)
Action = Tuple[str, Optional[str]]  # ("MOVE", dest) or ("REST", None)

def actions_mdp(loc: str) -> List[Action]:
    acts: List[Action] = [("REST", None)]
    for nb in neighbors(loc):
        acts.append(("MOVE", nb))
    return acts

def is_terminal(s: State) -> bool:
    t, _, _ = s
    return t == NUM_TIME_STEPS - 1  # terminal at 21:00 (last slot)

# Walk times (minutes) — simple and interpretable (tune)
# Use graph distance 1 (neighbor) = 4 min, REST = 0
MOVE_COST = 4.0
REST_COST = 0.0

# Expected waiting time model from busyness probability
def expected_wait(p_busy: float) -> float:
    # If busy: long wait; if not busy: short wait
    wait_busy, wait_not = 12.0, 3.0
    return p_busy * wait_busy + (1 - p_busy) * wait_not

# Mismatch penalty by type + destination (keep simple)
MISMATCH_PENALTY: Dict[Tuple[str, str], float] = {
    ("athlete", "VillagePark"): 0.0,
    ("athlete", "Gym"): 0.0,  # not in LOCATIONS now, left here for example
    ("athlete", "StudentCenter"): 2.0,
    ("athlete", "PlayHouse"): 4.0,
    ("athlete", "BoulevardStudios"): 4.0,
    ("athlete", "Library"): 2.0,
    ("athlete", "OffCampus"): 3.0,
}

def mismatch(student_type: str, loc: str) -> float:
    return MISMATCH_PENALTY.get((student_type, loc), 1.0)

# Practice readiness at 18:00 (time_step index = 5)
PRACTICE_T = TIME_SLOTS.index("18:00")
PRACTICE_LOCATION = "VillagePark"

def readiness_reward(t: int, loc: str, energy: int) -> float:
    """
    Big future objective:
    At 18:00, reward being at VillagePark with OK/HIGH energy.
    Penalize being low energy at 18:00 (even if you're there).
    """
    if t != PRACTICE_T:
        return 0.0
    if loc == PRACTICE_LOCATION and energy >= OK:
        return +12.0
    if energy == LOW:
        return -12.0
    # not at practice location, but not dead: still bad
    return -6.0

def transition_mdp(
    s: State,
    a: Action,
    p_busy_table: List[Dict[str, float]],
    student_type: str,
    rng: Optional[random.Random] = None,
) -> List[Tuple[float, State, float]]:
    """
    Returns list of (prob, s', reward)

    Stochastic movement:
    - MOVE succeeds w.p. 0.85 else stay (like a delay / distraction)
    - REST always stays

    Time always advances: t' = min(t+1, last)
    Energy dynamics:
    - MOVE tends to drain energy
    - REST tends to increase energy
    Crowdiness affects "wait cost" at the destination (or current loc if failed).
    """
    rng = rng or random
    t, loc, e = s
    t2 = min(t + 1, NUM_TIME_STEPS - 1)

    if a[0] == "REST":
        # energy recovers
        e2 = clamp_energy(e + 1)
        loc2 = loc
        p_busy = p_busy_table[t2][loc2]
        r = -(REST_COST + expected_wait(p_busy) + mismatch(student_type, loc2)) + readiness_reward(t2, loc2, e2)
        return [(1.0, (t2, loc2, e2), r)]

    if a[0] == "MOVE":
        dest = a[1]
        assert dest is not None
        # succeed
        e_succ = clamp_energy(e - 1)
        p_busy_succ = p_busy_table[t2][dest]
        r_succ = -(MOVE_COST + expected_wait(p_busy_succ) + mismatch(student_type, dest)) + readiness_reward(t2, dest, e_succ)

        # fail -> stay
        e_fail = clamp_energy(e - 1)  # still spent effort
        p_busy_fail = p_busy_table[t2][loc]
        r_fail = -(MOVE_COST + expected_wait(p_busy_fail) + mismatch(student_type, loc)) + readiness_reward(t2, loc, e_fail)

        return [
            (0.85, (t2, dest, e_succ), r_succ),
            (0.15, (t2, loc, e_fail), r_fail),
        ]

    raise ValueError("Unknown action")

def all_states_mdp() -> List[State]:
    S: List[State] = []
    for t in range(NUM_TIME_STEPS):
        for loc in LOCATIONS:
            for e in [LOW, OK, HIGH]:
                S.append((t, loc, e))
    return S

def value_iteration_mdp(
    p_busy_table: List[Dict[str, float]],
    student_type: str,
    gamma: float = 0.95,
    epsilon: float = 1e-4,
) -> Tuple[Dict[State, float], Dict[State, Action]]:
    """
    Standard value iteration on the augmented state space (includes time).
    Because time always moves forward, this MDP is episodic and will converge quickly.
    """
    S = all_states_mdp()
    U: Dict[State, float] = {s: 0.0 for s in S}

    while True:
        delta = 0.0
        U2 = dict(U)

        for s in S:
            if is_terminal(s):
                # terminal utility 0 (could also include final reward)
                U2[s] = 0.0
                continue

            t, loc, _ = s
            best = -float("inf")
            for a in actions_mdp(loc):
                exp = 0.0
                for p, sp, r in transition_mdp(s, a, p_busy_table, student_type):
                    exp += p * (r + gamma * U[sp])
                best = max(best, exp)

            U2[s] = best
            delta = max(delta, abs(U2[s] - U[s]))

        U = U2
        if delta < epsilon:
            break

    # Extract optimal policy
    pi: Dict[State, Action] = {}
    for s in S:
        if is_terminal(s):
            continue
        _, loc, _ = s
        best_a, best_val = None, -float("inf")
        for a in actions_mdp(loc):
            exp = 0.0
            for p, sp, r in transition_mdp(s, a, p_busy_table, student_type):
                exp += p * (r + gamma * U[sp])
            if exp > best_val:
                best_val = exp
                best_a = a
        assert best_a is not None
        pi[s] = best_a

    return U, pi

def pretty_action(a: Action) -> str:
    if a[0] == "REST":
        return "REST"
    return f"MOVE->{a[1]}"

def simulate_policy(
    pi: Dict[State, Action],
    p_busy_table: List[Dict[str, float]],
    student_type: str,
    start_state: State,
    seed: int = 0,
) -> List[Tuple[State, Action, float]]:
    rng = random.Random(seed)
    traj: List[Tuple[State, Action, float]] = []
    s = start_state
    while not is_terminal(s):
        a = pi[s]
        # sample transition
        dist = transition_mdp(s, a, p_busy_table, student_type)
        rnum = rng.random()
        cum = 0.0
        chosen = dist[-1]
        for item in dist:
            cum += item[0]
            if rnum <= cum:
                chosen = item
                break
        _, sp, r = chosen
        traj.append((s, a, r))
        s = sp
    return traj

## Part 3 — Sequential Planning as an MDP

This section formalizes the problem as a **Markov Decision Process (MDP)**.

The state is defined as:

\[
s = (t, loc, energy)
\]

where:
- \(t\) is the current time step,
- \(loc\) is the current campus location,
- \(energy\) is the current energy level.

The available actions are:
- **REST**, which keeps the agent in place and tends to improve energy,
- **MOVE→neighbor**, which moves the agent to an adjacent node in the campus graph.

This is a sequential planning problem because actions affect future opportunities. For example:
- moving now may reduce energy later,
- resting now may prepare the agent for an important future event,
- and choosing the wrong location can make later rewards harder to achieve.

A particularly important long-term objective is added at **18:00**, where the agent is rewarded for being at **VillagePark** with enough energy for practice. This creates a delayed reward structure. Because of that, maximizing immediate reward is no longer enough. An action must be judged by both its short-term effect and its future consequences.

The MDP transition model is stochastic:
- a movement action succeeds with probability 0.85,
- and fails with probability 0.15, in which case the agent stays in place.

The reward function combines several components:
- movement or rest cost,
- expected waiting time based on busyness,
- mismatch penalty between student type and destination,
- and practice readiness reward or penalty at the critical future time.

The algorithm used to solve this MDP is **value iteration**. It computes:
- the value of each state,
- and the optimal policy for each state.

This section therefore shows the full planning layer of the problem and illustrates clearly why a myopic greedy strategy can fail while a longer-term optimal policy succeeds.

In [4]:
# ============================================================
# Part 3 (17.3): Bandit version (unknown rewards)
# ============================================================

"""
We turn a slice of the problem into a bandit:
- At 10:00, the agent chooses ONE "hangout" destination (arm)
- Reward is a noisy satisfaction score (unknown mean initially)
- Compare Greedy vs ε-greedy vs UCB1

This is a bandit because:
- No state transitions considered inside the bandit (single-step choice)
- Rewards are unknown and must be learned from experience
"""

BANDIT_T = TIME_SLOTS.index("10:00")
ARMS = ["StudentCenter", "Library", "VillagePark", "PlayHouse"]

def true_mean_reward_from_model(
    arm: str,
    p_busy_table: List[Dict[str, float]],
    student_type: str,
) -> float:
    """
    Turn our cost model into a [0,1]-like reward.
    Lower cost => higher reward.
    This gives a consistent "ground truth" for the bandit.
    """
    p_busy = p_busy_table[BANDIT_T][arm]
    cost = MOVE_COST + expected_wait(p_busy) + mismatch(student_type, arm)
    # map cost to reward; tweakable
    # reward = exp(-cost/scale)
    scale = 15.0
    return math.exp(-cost / scale)

def pull_arm(arm: str, true_means: Dict[str, float], rng: random.Random) -> float:
    mu = true_means[arm]
    # noisy reward
    r = rng.gauss(mu, 0.06)
    return float(min(1.0, max(0.0, r)))

def run_greedy_bandit(T: int, true_means: Dict[str, float], seed: int = 0):
    rng = random.Random(seed)
    Q = {a: 0.0 for a in ARMS}
    N = {a: 0 for a in ARMS}
    rewards = []

    for _ in range(T):
        arm = max(ARMS, key=lambda a: (Q[a], rng.random()))
        r = pull_arm(arm, true_means, rng)
        N[arm] += 1
        Q[arm] += (r - Q[arm]) / N[arm]
        rewards.append(r)

    return rewards, N

def run_eps_greedy_bandit(T: int, true_means: Dict[str, float], eps: float, seed: int = 0):
    rng = random.Random(seed)
    Q = {a: 0.0 for a in ARMS}
    N = {a: 0 for a in ARMS}
    rewards = []

    for _ in range(T):
        if rng.random() < eps:
            arm = rng.choice(ARMS)
        else:
            arm = max(ARMS, key=lambda a: (Q[a], rng.random()))
        r = pull_arm(arm, true_means, rng)
        N[arm] += 1
        Q[arm] += (r - Q[arm]) / N[arm]
        rewards.append(r)

    return rewards, N

def run_ucb1_bandit(T: int, true_means: Dict[str, float], seed: int = 0):
    rng = random.Random(seed)
    Q = {a: 0.0 for a in ARMS}
    N = {a: 0 for a in ARMS}
    rewards = []

    # init: pull each once
    for a in ARMS:
        r = pull_arm(a, true_means, rng)
        Q[a] = r
        N[a] = 1
        rewards.append(r)

    for t in range(len(ARMS), T):
        def ucb(a: str) -> float:
            return Q[a] + math.sqrt(2.0 * math.log(t + 1) / N[a])

        arm = max(ARMS, key=ucb)
        r = pull_arm(arm, true_means, rng)
        N[arm] += 1
        Q[arm] += (r - Q[arm]) / N[arm]
        rewards.append(r)

    return rewards, N

## Part 4 — Bandit Learning: Exploration vs Exploitation

This section simplifies the larger sequential problem into a one-step learning problem at a single time slice.

At **10:00**, the agent chooses one destination among a small set of candidate locations. In this formulation, each possible destination becomes an **arm** in a multi-armed bandit. The reward is a noisy satisfaction value, and the true average reward of each arm is initially unknown to the agent.

This is different from the MDP setting because:
- there is no long-term state transition inside the bandit itself,
- the agent is not planning across multiple future time steps,
- and the main challenge is instead learning which option is best through repeated experience.

Three strategies are compared:

1. **Greedy**  
   Always picks the arm with the currently highest estimated value.  
   This can perform poorly if early random outcomes are misleading.

2. **ε-greedy**  
   Usually exploits the best current estimate, but with probability ε it explores a random arm.  
   This allows correction of early mistakes and helps discover better options.

3. **UCB1**  
   Uses an optimism-based exploration bonus. Arms with more uncertainty receive a temporary advantage, encouraging exploration in a more targeted way than random choice.

This section is useful because it isolates the exploration-exploitation tradeoff very clearly. In the MDP, planning and delayed rewards dominate the problem. In the bandit, uncertainty about reward estimates dominates the problem.

So while the bandit is simpler, it highlights an important principle: when rewards are initially unknown, some exploration is necessary to learn well and achieve better long-run performance.

In [5]:
# ============================================================
# Part 4 (17.4): Partial Observability (belief update)
# ============================================================

"""
Now we hide a variable:
- Hidden state: StudentCenterCrowd ∈ {Busy, NotBusy} at each time t.
Agent cannot see it directly.
Agent maintains belief b_t = P(Busy at StudentCenter at time t).

Actions now include CHECK (observation).
Belief update uses:
- transition: crowd can change over time
- observation: noisy report
"""

HIDDEN = ["Busy", "NotBusy"]
OBS = ["ReportBusy", "ReportNotBusy"]

# Observation model P(obs | hidden)
LIKELIHOOD = {
    ("ReportBusy", "Busy"): 0.85,
    ("ReportNotBusy", "Busy"): 0.15,
    ("ReportBusy", "NotBusy"): 0.15,
    ("ReportNotBusy", "NotBusy"): 0.85,
}

# Crowd transition model P(hidden' | hidden)
# crowd tends to persist, but can flip
CROWD_TRANS = {
    ("Busy", "Busy"): 0.70, ("Busy", "NotBusy"): 0.30,
    ("NotBusy", "Busy"): 0.25, ("NotBusy", "NotBusy"): 0.75,
}

def bayes_predict(prior_busy: float) -> float:
    # b'(Busy) = P(Busy|Busy)*b + P(Busy|NotBusy)*(1-b)
    return CROWD_TRANS[("Busy", "Busy")] * prior_busy + CROWD_TRANS[("NotBusy", "Busy")] * (1 - prior_busy)

def bayes_update(prior_busy: float, obs: str) -> float:
    """
    One-step filter: predict -> correct
    """
    pred = bayes_predict(prior_busy)
    p_o_busy = LIKELIHOOD[(obs, "Busy")]
    p_o_not = LIKELIHOOD[(obs, "NotBusy")]
    num = p_o_busy * pred
    den = num + p_o_not * (1 - pred)
    return pred if den == 0 else num / den

def p_obs_from_belief(prior_busy: float, obs: str) -> float:
    pred = bayes_predict(prior_busy)
    return LIKELIHOOD[(obs, "Busy")] * pred + LIKELIHOOD[(obs, "NotBusy")] * (1 - pred)


# ============================================================
# Part 5 (17.5): Approximate POMDP solving (rollout)
# ============================================================

"""
Exact POMDP is hard because belief is continuous.
Approximation used: Monte Carlo rollouts from belief.
At each decision:
- sample hidden crowd from belief
- simulate forward for short horizon
- evaluate actions by average discounted return
"""

POMDP_ACTIONS = ["REST", "MOVE", "CHECK_SC"]  # check student center
CHECK_COST = 2.0  # minutes / negative reward (like Week 2 info cost)

def sample_hidden_from_belief(b_busy: float, rng: random.Random) -> str:
    return "Busy" if rng.random() < b_busy else "NotBusy"

def sample_obs(hidden: str, rng: random.Random) -> str:
    p = LIKELIHOOD[("ReportBusy", hidden)]
    return "ReportBusy" if rng.random() < p else "ReportNotBusy"

def step_reward_pomdp(
    t2: int,
    loc2: str,
    e2: int,
    student_type: str,
    p_busy_table: List[Dict[str, float]],
    hidden_sc: str,
    action_label: str,
) -> float:
    """
    Reward uses same structure as MDP but:
    - StudentCenter busyness uses hidden_sc for wait time (more faithful POMDP)
    - other locations use p_busy_table
    - CHECK has explicit cost
    """
    if action_label == "CHECK_SC":
        # pay sensing cost, no movement reward in this step besides time passing
        return -(CHECK_COST)

    # determine p_busy used for waiting
    if loc2 == "StudentCenter":
        p_busy = 1.0 if hidden_sc == "Busy" else 0.0
    else:
        p_busy = p_busy_table[t2][loc2]

    # movement/rest cost
    base_cost = REST_COST if action_label == "REST" else MOVE_COST
    total_cost = base_cost + expected_wait(p_busy) + mismatch(student_type, loc2)

    return -(total_cost) + readiness_reward(t2, loc2, e2)

def pomdp_transition(
    t: int,
    loc: str,
    e: int,
    b_busy: float,
    hidden_sc: str,
    action: str,
    p_busy_table: List[Dict[str, float]],
    student_type: str,
    rng: random.Random,
) -> Tuple[int, str, int, float, str, Optional[str], float]:
    """
    Returns:
      (t2, loc2, e2, b2, hidden2, obs, reward)

    - time always advances
    - hidden crowd evolves by CROWD_TRANS
    - CHECK gives an observation and belief update
    """
    t2 = min(t + 1, NUM_TIME_STEPS - 1)

    # evolve hidden crowd
    p_stay_busy = CROWD_TRANS[(hidden_sc, "Busy")]
    hidden2 = "Busy" if rng.random() < p_stay_busy else "NotBusy"

    obs = None

    if action == "CHECK_SC":
        # observe (based on hidden2) and update belief
        obs = sample_obs(hidden2, rng)
        b2 = bayes_update(b_busy, obs)
        # location, energy unchanged (but time advanced)
        loc2 = loc
        e2 = e
    elif action == "REST":
        b2 = bayes_predict(b_busy)
        loc2 = loc
        e2 = clamp_energy(e + 1)
    elif action == "MOVE":
        b2 = bayes_predict(b_busy)
        # choose a neighbor randomly for rollout default (we’ll choose in the planner)
        # Here we will override dest outside this function if needed.
        raise ValueError("Use pomdp_transition_move(...) for MOVE with chosen dest.")
    else:
        raise ValueError("bad action")

    r = step_reward_pomdp(t2, loc2, e2, student_type, p_busy_table, hidden2, action)
    return t2, loc2, e2, b2, hidden2, obs, r

def pomdp_transition_move(
    t: int,
    loc: str,
    e: int,
    b_busy: float,
    hidden_sc: str,
    dest: str,
    p_busy_table: List[Dict[str, float]],
    student_type: str,
    rng: random.Random,
) -> Tuple[int, str, int, float, str, float]:
    """
    MOVE with chosen destination neighbor:
    - succeed w.p.0.85 else stay
    - energy -1
    - belief predict only (no observation)
    """
    t2 = min(t + 1, NUM_TIME_STEPS - 1)

    # evolve hidden crowd
    p_stay_busy = CROWD_TRANS[(hidden_sc, "Busy")]
    hidden2 = "Busy" if rng.random() < p_stay_busy else "NotBusy"

    b2 = bayes_predict(b_busy)
    e2 = clamp_energy(e - 1)

    if rng.random() < 0.85:
        loc2 = dest
    else:
        loc2 = loc

    r = step_reward_pomdp(t2, loc2, e2, student_type, p_busy_table, hidden2, action_label="MOVE")
    return t2, loc2, e2, b2, hidden2, r

def rollout_value(
    start_t: int,
    start_loc: str,
    start_e: int,
    start_b: float,
    first_action: Tuple[str, Optional[str]],
    p_busy_table: List[Dict[str, float]],
    student_type: str,
    horizon: int,
    gamma: float,
    rng: random.Random,
) -> float:
    """
    Approximate Q(b, a) by simulation.
    first_action:
      ("REST", None) or ("CHECK_SC", None) or ("MOVE", dest)
    After first step, use a simple default policy:
      - if time is near 18:00, drift toward VillagePark
      - if energy low, REST
      - else move toward VillagePark with random neighbor step
    """
    # sample initial hidden
    hidden = sample_hidden_from_belief(start_b, rng)
    t, loc, e, b = start_t, start_loc, start_e, start_b

    total = 0.0
    disc = 1.0

    # apply first action
    if first_action[0] == "REST":
        t, loc, e, b, hidden, _, r = pomdp_transition(t, loc, e, b, hidden, "REST", p_busy_table, student_type, rng)
    elif first_action[0] == "CHECK_SC":
        t, loc, e, b, hidden, _, r = pomdp_transition(t, loc, e, b, hidden, "CHECK_SC", p_busy_table, student_type, rng)
    elif first_action[0] == "MOVE":
        dest = first_action[1]
        assert dest is not None
        t, loc, e, b, hidden, r = pomdp_transition_move(t, loc, e, b, hidden, dest, p_busy_table, student_type, rng)
    else:
        raise ValueError("bad first action")

    total += disc * r
    disc *= gamma

    # default policy for remaining steps
    for _ in range(horizon - 1):
        if t >= NUM_TIME_STEPS - 1:
            break

        # If low energy and not at the very end, REST
        if e == LOW:
            t, loc, e, b, hidden, _, r = pomdp_transition(t, loc, e, b, hidden, "REST", p_busy_table, student_type, rng)
        else:
            # Move roughly toward VillagePark:
            # if already at VillagePark, REST to keep energy high
            if loc == PRACTICE_LOCATION:
                t, loc, e, b, hidden, _, r = pomdp_transition(t, loc, e, b, hidden, "REST", p_busy_table, student_type, rng)
            else:
                nbs = neighbors(loc)
                if not nbs:
                    t, loc, e, b, hidden, _, r = pomdp_transition(t, loc, e, b, hidden, "REST", p_busy_table, student_type, rng)
                else:
                    # heuristic: prefer neighbor that is VillagePark or closer (simple)
                    if PRACTICE_LOCATION in nbs:
                        dest = PRACTICE_LOCATION
                    else:
                        dest = rng.choice(nbs)
                    t, loc, e, b, hidden, r = pomdp_transition_move(t, loc, e, b, hidden, dest, p_busy_table, student_type, rng)

        total += disc * r
        disc *= gamma

    return total

def decide_pomdp_action(
    t: int,
    loc: str,
    e: int,
    b_busy: float,
    p_busy_table: List[Dict[str, float]],
    student_type: str,
    rollouts: int = 600,
    horizon: int = 5,
    gamma: float = 0.95,
    seed: int = 0,
) -> Tuple[Tuple[str, Optional[str]], Dict[str, float]]:
    rng = random.Random(seed)

    # Build candidate actions from graph
    candidates: List[Tuple[str, Optional[str]]] = [("REST", None), ("CHECK_SC", None)]
    for nb in neighbors(loc):
        candidates.append(("MOVE", nb))

    values: Dict[str, float] = {}
    for a in candidates:
        s = 0.0
        for _ in range(rollouts):
            s += rollout_value(t, loc, e, b_busy, a, p_busy_table, student_type, horizon, gamma, rng)
        values[pretty_action(a)] = s / rollouts

    best_label = max(values, key=values.get)

    # map label back to action tuple
    label_to_action = {pretty_action(a): a for a in candidates}
    return label_to_action[best_label], values

## Part 5 — Partial Observability and Approximate POMDP Planning

This section adds another level of realism: the agent no longer fully knows the environment.

The hidden variable is the crowd condition of **StudentCenter**:
- Busy
- NotBusy

The agent cannot directly observe this state. Instead, it maintains a **belief**:

\[
b_t = P(\text{Busy at StudentCenter at time } t)
\]

That belief is updated using a Bayes filter with two components:

1. **Prediction step**  
   The hidden crowd condition evolves over time according to a transition model.  
   This captures the idea that if StudentCenter is busy now, it is somewhat likely to remain busy next step, but it may also change.

2. **Correction step**  
   If the agent takes a checking action, it receives a noisy observation:
   - `ReportBusy`
   - `ReportNotBusy`

   Since observations are imperfect, the agent does not jump directly to certainty. Instead, it updates its probability belief using Bayes’ rule.

This converts the problem from an MDP into a **POMDP** because the state is no longer fully observable.

However, exact POMDP solving is difficult. The belief is continuous, which greatly increases the complexity of the state space. To make the problem tractable, this notebook uses an **approximate rollout method**.

The idea of the approximation is:
- sample a hidden crowd state from the current belief,
- simulate possible futures for a short horizon,
- estimate discounted return for each candidate action,
- and choose the action with the highest average rollout value.

This is a practical compromise:
- it is much simpler than exact POMDP planning,
- but it still captures the effect of hidden state, sensing, and uncertainty.

So this section shows how the strategy changes when the agent must act not only under risk, but also under incomplete information.

In [6]:
# ============================================================
# DEMO 
# ============================================================

if __name__ == "__main__":
    print("=== Building belief table p_busy[L,t] from Week1-style Monte Carlo ===")
    p_busy_table = estimate_p_busy_table(
        num_traces=2000,
        busy_threshold=0.18,
        friend_boost=2,
        seed=7,
    )

    t_demo = TIME_SLOTS.index("10:00")
    print(f"\nBelief snapshot at {TIME_SLOTS[t_demo]} (t={t_demo}):")
    for L in LOCATIONS:
        print(f"  P(Busy({L}) at {TIME_SLOTS[t_demo]}) ≈ {p_busy_table[t_demo][L]:.3f}")

    # ---------------- Part 2: MDP ----------------
    print("\n\n==============================")
    print("PART 2: MDP + Value Iteration")
    print("==============================")

    gamma = 0.95
    U, pi = value_iteration_mdp(p_busy_table, student_type=AGENT_TYPE, gamma=gamma, epsilon=1e-4)

    # Choose a start state for demonstration:
    # Start at 10:00, StudentCenter, OK energy
    start_state: State = (t_demo, "StudentCenter", OK)

    print(f"\nStart state s0 = (time={TIME_SLOTS[start_state[0]]}, loc={start_state[1]}, energy={start_state[2]})")
    print("Optimal first action π*(s0) =", pretty_action(pi[start_state]))
    print("State value V(s0) =", round(U[start_state], 3))

    traj = simulate_policy(pi, p_busy_table, AGENT_TYPE, start_state, seed=7)
    print("\nOne simulated optimal trajectory (state -> action -> reward):")
    for (s, a, r) in traj:
        t, loc, e = s
        print(f"  t={TIME_SLOTS[t]}  loc={loc:18s}  E={e}  -> {pretty_action(a):18s}  reward={r: .3f}")

    # Greedy baseline (myopic):
    def greedy_action(s: State) -> Action:
        t, loc, _ = s
        best_a, best_r = None, -float("inf")
        for a in actions_mdp(loc):
            # greedy = maximize immediate expected reward ONLY (ignore gamma*U(s'))
            exp_r = 0.0
            for p, sp, r in transition_mdp(s, a, p_busy_table, AGENT_TYPE):
                exp_r += p * r
            if exp_r > best_r:
                best_r = exp_r
                best_a = a
        assert best_a is not None
        return best_a

    print("\n--- Why greedy can fail (myopic vs optimal) ---")
    s = start_state
    rng = random.Random(7)
    for step in range(3):
        a_opt = pi[s]
        a_greedy = greedy_action(s)
        print(f"At state (t={TIME_SLOTS[s[0]]}, loc={s[1]}, E={s[2]})")
        print(f"  optimal: {pretty_action(a_opt)}")
        print(f"  greedy:  {pretty_action(a_greedy)}")
        # advance one step using greedy to show divergence
        dist = transition_mdp(s, a_greedy, p_busy_table, AGENT_TYPE)
        rnum = rng.random()
        cum = 0.0
        chosen = dist[-1]
        for item in dist:
            cum += item[0]
            if rnum <= cum:
                chosen = item
                break
        _, s_next, r = chosen
        s = s_next
        if is_terminal(s):
            break

    # ---------------- Part 3: Bandit ----------------
    print("\n\n==============================")
    print("PART 3: Multi-Armed Bandit")
    print("==============================")

    true_means = {a: true_mean_reward_from_model(a, p_busy_table, AGENT_TYPE) for a in ARMS}
    print("True (hidden) mean rewards (unknown to agent):")
    for a in ARMS:
        print(f"  {a:14s}: {true_means[a]:.3f}")

    T = 400
    r_g, counts_g = run_greedy_bandit(T, true_means, seed=7)
    r_e, counts_e = run_eps_greedy_bandit(T, true_means, eps=0.1, seed=7)
    r_u, counts_u = run_ucb1_bandit(T, true_means, seed=7)

    def avg(xs): return sum(xs) / len(xs)

    print(f"\nAverage reward over T={T} pulls:")
    print(f"  Greedy:     {avg(r_g):.3f}  picks={counts_g}")
    print(f"  ε-greedy:   {avg(r_e):.3f}  picks={counts_e}  (eps=0.1)")
    print(f"  UCB1:       {avg(r_u):.3f}  picks={counts_u}")

    # ---------------- Part 4/5: POMDP ----------------
    print("\n\n==============================")
    print("PART 4/5: POMDP (belief + approximate solving)")
    print("==============================")

    print("Hidden variable: StudentCenterCrowd ∈ {Busy, NotBusy}")
    print("Belief state: b = P(Busy at StudentCenter)")
    print("Observation model: P(ReportBusy | Busy)=0.85, P(ReportBusy | NotBusy)=0.15")
    print(f"Check cost: {CHECK_COST} (minutes-equivalent)")

    # Prior belief about StudentCenter crowd at 10:00 from Week1 table:
    b0 = p_busy_table[t_demo]["StudentCenter"]
    print(f"\nPrior belief at 10:00: b0 = P(Busy(StudentCenter)) ≈ {b0:.3f}")

    # Decide using approximate POMDP rollouts from belief
    a_star, vals = decide_pomdp_action(
        t=t_demo,
        loc="StudentCenter",
        e=OK,
        b_busy=b0,
        p_busy_table=p_busy_table,
        student_type=AGENT_TYPE,
        rollouts=600,
        horizon=5,
        gamma=0.95,
        seed=7,
    )

    print("\nEstimated action values from rollouts (higher is better):")
    for k, v in sorted(vals.items(), key=lambda x: x[1], reverse=True):
        print(f"  {k:18s}: {v:.3f}")

    print("\nApproximate best POMDP action =", pretty_action(a_star))

    print("\n--- Markdown-ready explanation (paste into notebook) ---\n")
    print(f"""
### Part 1 (Sequential decision problem)
- **State**: \\(s=(t,\\,loc,\\,energy)\\) where:
  - \\(t\\) is one of {TIME_SLOTS}
  - \\(loc\\in\\) campus locations
  - \\(energy\\in\\{{Low,OK,High\\}}\\)
- **Actions**:
  - **REST** (stay put, increases energy)
  - **MOVE→neighbor** (only along the campus graph)
- **Why greedy may fail**:
  - Greedy maximizes immediate reward but ignores future penalties.
  - We add a strong future objective at **18:00**: being at **{PRACTICE_LOCATION}** with OK/HIGH energy.
  - Therefore an optimal policy may rest earlier or move earlier to arrive ready, even if that is not the best immediate choice.

### Part 2 (MDP + Value Iteration)
- **Transition model** (probabilistic):
  - MOVE succeeds with probability 0.85, otherwise the agent stays (delay).
  - Time always advances to the next slot.
  - Energy decreases with MOVE, increases with REST.
- **Reward** (multi-attribute):
  - Negative total time cost = walking + expected waiting + mismatch penalty
  - Plus a **practice readiness bonus/penalty at 18:00**
- We compute **state values** \\(V(s)\\) and an **optimal policy** \\(\\pi^*(s)\\) using **value iteration** with \\(\\gamma={gamma}\\).

### Part 3 (Bandit)
- Arms = {ARMS}
- Rewards are noisy and **unknown** to the agent.
- We compare Greedy vs ε-greedy vs UCB1 to show exploration vs exploitation tradeoff.

### Part 4 (Partial Observability)
- Hidden state: **StudentCenterCrowd ∈ {{Busy, NotBusy}}**
- Belief state: \\(b_t = P(Busy)\\)
- We update beliefs using a Bayes filter:
  - prediction step (Markov crowd dynamics)
  - correction step (noisy observation)

### Part 5 (Approximate POMDP solving)
- Exact POMDP solutions are infeasible because beliefs are continuous and the state space explodes.
- Approximation used: **Monte Carlo rollouts**:
  - sample hidden crowd from belief
  - simulate forward for a short horizon
  - estimate discounted return and pick the best action
""".strip())

=== Building belief table p_busy[L,t] from Week1-style Monte Carlo ===

Belief snapshot at 10:00 (t=1):
  P(Busy(LawrenceHall) at 10:00) ≈ 0.013
  P(Busy(ThayerHall) at 10:00) ≈ 0.013
  P(Busy(WestPenn) at 10:00) ≈ 0.015
  P(Busy(StudentCenter) at 10:00) ≈ 0.490
  P(Busy(Library) at 10:00) ≈ 0.253
  P(Busy(VillagePark) at 10:00) ≈ 0.363
  P(Busy(PlayHouse) at 10:00) ≈ 0.318
  P(Busy(BoulevardStudios) at 10:00) ≈ 0.324
  P(Busy(BoulevardAppartments) at 10:00) ≈ 0.032
  P(Busy(OffCampus) at 10:00) ≈ 0.014


PART 2: MDP + Value Iteration

Start state s0 = (time=10:00, loc=StudentCenter, energy=1)
Optimal first action π*(s0) = MOVE->LawrenceHall
State value V(s0) = -30.088

One simulated optimal trajectory (state -> action -> reward):
  t=10:00  loc=StudentCenter       E=1  -> MOVE->LawrenceHall  reward=-8.135
  t=12:00  loc=LawrenceHall        E=0  -> REST                reward=-4.085
  t=14:00  loc=LawrenceHall        E=1  -> REST                reward=-4.139
  t=16:00  loc=LawrenceHall 

## Results and Interpretation

The output of this notebook allows comparison across several levels of intelligent decision-making.

First, the Monte Carlo simulation produces a belief table for campus crowd levels over time. This gives the agent a probabilistic model of busyness rather than assuming full certainty.

Second, the MDP solution shows how optimal decisions differ from greedy ones when future consequences matter. The presence of energy dynamics and a delayed practice reward makes planning over time essential.

Third, the bandit experiments show that when reward values are initially unknown, exploration can outperform pure exploitation over the long run. This highlights the importance of learning before committing too strongly.

Finally, the POMDP section shows how uncertainty about hidden state changes the strategy further. The agent must now reason over beliefs instead of directly observed truth, and the checking action becomes useful when information can improve later action quality enough to justify its cost.

Together, these results illustrate that intelligent action selection depends on:
- the structure of the environment,
- uncertainty about outcomes,
- delayed rewards,
- information costs,
- and the computational method used to solve the problem.

## Answers to the Required Questions

### How did planning over time change decisions?

Planning over time changed decisions by making the agent care about future consequences rather than only immediate rewards. In the MDP setting, an action was evaluated not just by what it produced right away, but also by how it affected later states such as future location, remaining energy, and readiness for the important 18:00 practice objective.

This created situations where the optimal policy chose actions that looked worse in the short term but led to better outcomes later. For example, resting earlier could be better than moving immediately if it preserved enough energy to arrive at VillagePark in a better condition at the practice time. Similarly, moving toward a strategically useful location earlier could be better than choosing a locally attractive option that left the agent poorly positioned later.

In this way, planning over time transformed the problem from “What gives the best immediate reward?” into “What sequence of actions leads to the best overall return?” That is exactly why the optimal MDP policy could differ from a greedy strategy.

### When did exploration improve performance?

Exploration improved performance in the bandit setting when the agent did not yet know the true average reward of each option. At the beginning, all reward estimates were uncertain, so committing too early to one arm could be risky.

A purely greedy strategy could get stuck exploiting an arm that only looked good because of favorable early noise. By contrast, ε-greedy improved performance by occasionally trying other actions, which gave the agent a chance to discover better options and correct bad initial estimates. UCB1 improved performance in an even more focused way by favoring arms with high uncertainty, balancing what looked promising with what still needed more evidence.

So exploration was most useful early in learning and whenever uncertainty about arm quality remained high. Once the best arm became clear, exploitation mattered more. But without enough exploration first, the agent could easily settle on a suboptimal action.

### How did partial observability affect strategy?

Partial observability changed the strategy by forcing the agent to reason about probabilities instead of directly observed facts. In the POMDP section, the crowd status of StudentCenter was hidden, so the agent could no longer condition its action on the true state. Instead, it had to maintain a belief \( b_t = P(Busy) \).

Because the hidden state evolved over time and observations were noisy, the agent also had to update that belief dynamically using a Bayes filter. This made the strategy more cautious and information-sensitive. Rather than simply asking “Is StudentCenter busy?”, the agent had to ask “How likely is StudentCenter to be busy, and is it worth paying to learn more before acting?”

This also introduced a new type of action: the information-gathering action `CHECK_SC`. The value of that action depended on whether the expected improvement in future decision quality was large enough to justify the sensing cost. So partial observability changed the strategy by making belief management and information gathering part of the decision process itself.

### What trade-offs did you make using approximate POMDP solutions?

The main trade-off in using approximate POMDP solutions was between computational tractability and exact optimality.

Exact POMDP solving is difficult because the belief state is continuous and the number of possible future belief trajectories grows very quickly. To avoid that explosion in complexity, this notebook used Monte Carlo rollouts over a limited horizon. That made the problem feasible to solve and still allowed the model to account for hidden state, noisy observations, and discounted future rewards.

However, that approximation came with limitations:
- the solution depends on sampled rollouts rather than exact evaluation,
- action values may contain simulation noise,
- the planning horizon is short, so very long-term effects may be underrepresented,
- and the quality of the result depends on the default rollout policy used after the first action.

So the trade-off was clear: the approximate approach was practical and flexible, but it sacrificed some precision and optimality. It is a reasonable choice when exact POMDP methods are too expensive, especially for demonstrating the main ideas of belief-based decision-making.

## Connection to Sequential Decision-Making

This notebook demonstrates how increasingly realistic decision models can be built step by step.

The campus graph and energy model create a sequential environment where actions have delayed effects. The MDP formulation shows how optimal planning must account for those long-term consequences. The bandit formulation isolates the exploration-exploitation tradeoff when rewards are unknown. The partial observability section introduces hidden state and belief updating. Finally, the approximate POMDP section shows how uncertainty and sensing can be integrated into planning even when exact solutions are computationally difficult.

Overall, the notebook illustrates several core ideas from sequential decision-making:
- optimal planning is different from greedy action selection,
- uncertainty changes how rewards must be evaluated,
- exploration can be necessary for long-run success,
- hidden state requires belief-based reasoning,
- and approximate methods are often needed when exact decision-making becomes too expensive.

That is why this project is a strong example of how MDPs, bandits, and POMDPs connect within one unified decision-theoretic framework.